# Wine Quality: Which Physicochemical Properties Relate to Sensory Scores?

Statistical analysis of ~6,500 Portuguese *vinho verde* wines (red and white) using hypothesis tests, ANOVA, linear regression and logistic regression.

**Question:** which physicochemical measurements are associated with higher sensory quality scores, and how well can a simple model separate good wines from the rest?

**Notebook structure**
1. Setup and data loading
2. Data quality checks
3. Exploratory analysis
4. Hypothesis testing (Welch t-test, Mann-Whitney, ANOVA, Tukey HSD)
5. Linear regression with diagnostics
6. Logistic regression (inference and prediction)
7. Conclusions and limitations

Cells marked **TODO** are for your own interpretation. Write them after you see your own results.

## 1. Setup and data loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.diagnostic import het_breuschpagan
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay
from ucimlrepo import fetch_ucirepo

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

In [ ]:
wine = fetch_ucirepo(id=186)          # UCI Wine Quality
df = wine.data.original.copy()        # features + quality + color
df.columns = [c.lower().replace(" ", "_") for c in df.columns]

FEATURES = ["fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar",
            "chlorides", "free_sulfur_dioxide", "total_sulfur_dioxide",
            "density", "ph", "sulphates", "alcohol"]

print(df.shape)
print(df.columns.tolist())   # check that the names above match
df.head()

## 2. Data quality checks

In [ ]:
print(df.dtypes, "\n")
print("Missing values per column:\n", df.isna().sum(), "\n")
print("Duplicated rows:", df.duplicated().sum())
print(df["color"].value_counts(), "\n")
print(df["quality"].value_counts().sort_index())

**TODO:** decide whether to keep or drop duplicated rows and write down why. Duplicates can inflate confidence in the tests below, so state your choice clearly (and, if you want to be thorough, repeat the main result both ways).

In [ ]:
# df = df.drop_duplicates().reset_index(drop=True)   # uncomment if you decide to drop them
df.describe().T

## 3. Exploratory analysis

In [ ]:
df[FEATURES].hist(bins=30, figsize=(14, 10))
plt.suptitle("Distribution of physicochemical variables", y=1.0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[FEATURES + ["quality"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x="quality", y="alcohol", hue="color", ax=axes[0])
sns.boxplot(data=df, x="quality", y="volatile_acidity", hue="color", ax=axes[1])
plt.tight_layout()
plt.show()

**TODO:** in 3 to 4 sentences, describe what you see: skewed variables, outliers, the strongest correlations with quality, and any pair of predictors that are strongly correlated with each other (this matters for the regression).

## 4. Hypothesis testing

### 4.1 Do red and white wines differ in alcohol content?
H0: mean alcohol is the same in red and white wines.

In [ ]:
# check the labels first: df["color"].unique()
red = df.loc[df["color"] == "red", "alcohol"]
white = df.loc[df["color"] == "white", "alcohol"]

print("Levene (equal variances):", stats.levene(red, white))
print("Welch t-test:", stats.ttest_ind(red, white, equal_var=False))
print("Mann-Whitney U:", stats.mannwhitneyu(red, white))

def cohens_d(a, b):
    pooled_sd = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / pooled_sd

print("Cohen's d:", round(cohens_d(red, white), 3))

**TODO:** with this many samples almost any difference is significant. Report the effect size (Cohen's d) and the difference in means, not only the p-value.

### 4.2 Does alcohol differ across quality groups?
Quality is grouped into low (5 or below), medium (6) and high (7 or above). H0: mean alcohol is the same in all three groups.

In [ ]:
df["quality_group"] = pd.cut(df["quality"], bins=[0, 5, 6, 10], labels=["low", "medium", "high"])
print(df["quality_group"].value_counts())

anova_model = smf.ols("alcohol ~ C(quality_group)", data=df).fit()
print(sm.stats.anova_lm(anova_model, typ=2))

print(pairwise_tukeyhsd(df["alcohol"], df["quality_group"]))

In [ ]:
# ANOVA assumptions: residual normality and equal variances
sm.qqplot(anova_model.resid, line="45", fit=True)
plt.title("ANOVA residuals Q-Q plot")
plt.show()
print("Levene across groups:", stats.levene(*[g["alcohol"].values for _, g in df.groupby("quality_group", observed=True)]))
print("Kruskal-Wallis (non-parametric check):", stats.kruskal(*[g["alcohol"].values for _, g in df.groupby("quality_group", observed=True)]))

**TODO:** state whether the ANOVA assumptions hold, and whether the Kruskal-Wallis result agrees with the ANOVA.

## 5. Linear regression with diagnostics

In [ ]:
formula = "quality ~ " + " + ".join(FEATURES) + " + C(color)"
ols = smf.ols(formula, data=df).fit()
print(ols.summary())

In [ ]:
# Multicollinearity
X_vif = sm.add_constant(df[FEATURES])
vif = pd.DataFrame({"feature": X_vif.columns,
                    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]})
vif[vif["feature"] != "const"].sort_values("VIF", ascending=False)

In [ ]:
# Residual diagnostics
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(ols.fittedvalues, ols.resid, alpha=0.3)
axes[0].axhline(0, color="red")
axes[0].set_xlabel("Fitted values"); axes[0].set_ylabel("Residuals"); axes[0].set_title("Residuals vs fitted")
sm.qqplot(ols.resid, line="45", fit=True, ax=axes[1])
axes[1].set_title("Residuals Q-Q plot")
plt.tight_layout()
plt.show()

bp = het_breuschpagan(ols.resid, ols.model.exog)
print("Breusch-Pagan p-value:", bp[1])

In [ ]:
# Standardised coefficients: comparable effect sizes across variables
df_z = df.copy()
df_z[FEATURES] = (df[FEATURES] - df[FEATURES].mean()) / df[FEATURES].std()
ols_z = smf.ols(formula, data=df_z).fit()
coefs = ols_z.params.drop("Intercept").sort_values()
coefs.plot(kind="barh", figsize=(8, 6), title="Standardised OLS coefficients")
plt.show()

**TODO:** report R-squared, the three most influential variables, the highest VIF and what you did about it (for example, checking the model with and without density), and whether the residual plots show problems. Quality is an ordinal score, so also state why a linear model is only an approximation here.

## 6. Logistic regression

Target: **good wine** = quality of 7 or above. The original `quality` column is removed from the predictors to avoid leakage.

In [ ]:
df["good"] = (df["quality"] >= 7).astype(int)
print(df["good"].value_counts(normalize=True))

X = df[FEATURES]
y = df["good"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

In [ ]:
# Inference: odds ratios per one standard deviation increase (statsmodels)
X_train_z = (X_train - X_train.mean()) / X_train.std()
logit = sm.Logit(y_train, sm.add_constant(X_train_z)).fit(disp=False)

odds = np.exp(pd.concat([logit.params, logit.conf_int()], axis=1))
odds.columns = ["odds_ratio", "ci_low", "ci_high"]
odds["p_value"] = logit.pvalues
odds.drop("const").sort_values("odds_ratio")

In [ ]:
# Prediction: evaluate on held-out data (scikit-learn)
clf = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print("ROC AUC:", round(roc_auc_score(y_test, proba), 3))
print(confusion_matrix(y_test, pred))
RocCurveDisplay.from_predictions(y_test, proba)
plt.show()

**TODO:** interpret the odds ratios in plain language (for example, "one standard deviation more alcohol multiplies the odds of a good wine by ..."). Comment on precision and recall for the *good* class, not only accuracy, and explain why accuracy alone would be misleading with imbalanced classes.

## 7. Conclusions and limitations

**TODO:** write 3 to 5 findings with your own numbers. Suggested points to cover:
- Which variables are most associated with quality, and in which direction
- How well a simple model separates good wines (use recall and precision for the good class)
- Where the tests and the regression agree or disagree

**Limitations to state honestly**
- Quality is a subjective, ordinal sensory score, not a measurement
- Classes are imbalanced (few excellent and few poor wines)
- Results describe associations, not causes
- Data covers only *vinho verde* wines from one region
- Possible duplicated rows and correlated predictors (see sections 2 and 5)